<a href="https://colab.research.google.com/github/mazzarrella0/AI-Mod-3/blob/main/Mod_3_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import zipfile
import os

zip_path = '/content/praktikum-modul-3-ai-2026.zip'
extract_path = '/content/modul3'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Cek struktur hasil ekstrak
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files[:5]:
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... ({len(files)} files total)')

modul3/
  sample_submission.csv
  test/
    img_13ad604a39c0.jpg
    img_bee1f60fd285.jpg
    img_c93323ce03b9.jpg
    img_f9e000380015.jpg
    img_10854ff29b8d.jpg
    ... (1200 files total)
  train/
    SeaLake/
    Forest/
    Highway/
    AnnualCrop/
    River/
    Industrial/
    HerbaceousVegetation/
    Residential/
    PermanentCrop/
    Pasture/


In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configuration
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
TRAIN_DIR = '/content/modul3/train'

# Data Augmentation and Preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 4495 images belonging to 10 classes.
Found 1118 images belonging to 10 classes.


In [16]:
from tensorflow.keras import layers, models

# Building a CNN Model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax') # Assuming 10 classes from the directory list
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 684,490 (2.61 MB)

 Trainable params: 684,490 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Training the model
epochs = 20
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)

Epoch 1/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 29s 197ms/step - accuracy: 0.2516 - loss: 1.9505 - val_accuracy: 0.3945 - val_loss: 1.6355
Epoch 2/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 30s 214ms/step - accuracy: 0.3918 - loss: 1.5727 - val_accuracy: 0.4544 - val_loss: 1.3487
Epoch 3/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 27s 191ms/step - accuracy: 0.4354 - loss: 1.4732 - val_accuracy: 0.4553 - val_loss: 1.3440
Epoch 4/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 27s 194ms/step - accuracy: 0.4803 - loss: 1.3414 - val_accuracy: 0.5894 - val_loss: 1.2201
Epoch 5/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 28s 196ms/step - accuracy: 0.5290 - loss: 1.2638 - val_accuracy: 0.5859 - val_loss: 1.1275
Epoch 6/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 28s 195ms/step - accuracy: 0.5744 - loss: 1.1881 - val_accuracy: 0.5680 - val_loss: 1.1318
Epoch 7/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 28s 202ms/step - accuracy: 0.5973 - loss: 1.1441 - val_accuracy: 0.6055 - val_loss: 1.0886
Epoch 8/20
141/141 ━━━━━━━━━━━━━━━━━━━━ 29s 203ms/step - accuracy: 0.6109 - loss: 1

In [18]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing import image

# Predicting test data for submission
test_dir = '/content/modul3/test'
test_files = sorted(os.listdir(test_dir))
predictions = []

# Get class mapping
class_map = {v: k for k, v in train_generator.class_indices.items()}

for file in test_files:
    img_path = os.path.join(test_dir, file)
    img = image.load_img(img_path, target_size=IMG_SIZE)
    x = image.img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    pred = model.predict(x, verbose=0)
    class_idx = np.argmax(pred)
    predictions.append(class_map[class_idx])

# Create submission
submission = pd.DataFrame({'image': test_files, 'label': predictions})
submission.to_csv('my_submission.csv', index=False)
print("Submission file created: my_submission.csv")

Submission file created: my_submission.csv
